# PROFE 2026 — Matching · v3 (surgical improvements over v1)

Starts from the v1 baseline (66.67%) and applies only two surgical changes:
1. Re-run the 3 degenerate exercises with text-only N=3 self-consistency.
2. Run VL **only** on exercises where SET1 (the content/option side) has images — text-only was literally blind on those.

Everything else from v1 is preserved unchanged.


## 1. Install

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece qwen-vl-utils

## 2. Mount Drive · paths · upload existing predictions if needed

Place `predictions_matching.json` (the 189-exercise file you already have) **in the test-set folder on Drive** (`MyDrive/PROFE26_test_set/predictions_matching.json`) before running this. If it's not there, this notebook will start from scratch — but you don't want that.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil
TEST_SET_DIR = '/content/drive/MyDrive/PROFE26_test_set'
MATCHING_JSON = os.path.join(TEST_SET_DIR, 'matching_dataset.json')
IMG_DIR_DRIVE = os.path.join(TEST_SET_DIR, 'img')

LOCAL_PRED  = '/content/predictions_matching.json'
DRIVE_PRED  = os.path.join(TEST_SET_DIR, 'predictions_matching.json')
OUTPUT_FILE = LOCAL_PRED  # save here every step; mirror to Drive after each pass

assert os.path.exists(MATCHING_JSON), f'Missing test set: {MATCHING_JSON}'

# === Find an existing predictions_matching.json anywhere reasonable ===
# We accept the file in: the test-set folder (canonical), Drive root, or /content.
# Whichever copy has the MOST exercises wins.
CANDIDATES = [
    DRIVE_PRED,
    LOCAL_PRED,
    '/content/drive/MyDrive/predictions_matching.json',
    '/content/drive/MyDrive/predictions_matching (1).json',
    '/content/predictions_matching (1).json',
]
best, best_count, best_path = None, -1, None
for path in CANDIDATES:
    if os.path.exists(path):
        try:
            with open(path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            if isinstance(data, dict) and len(data) > best_count:
                best, best_count, best_path = data, len(data), path
        except Exception as e:
            print(f'  (skipped {path}: {e})')

if best is not None:
    with open(LOCAL_PRED, 'w', encoding='utf-8') as f:
        json.dump(best, f, ensure_ascii=False, indent=2)
    try:
        shutil.copy(LOCAL_PRED, DRIVE_PRED)
    except Exception as e:
        print('  (could not mirror to Drive:', e, ')')
    print(f'Existing predictions loaded: {best_count} exercises (from {best_path}).')
else:
    print('No existing predictions found anywhere - first run.')

# === Copy images to /content for fast access (Drive over FUSE is slow per-file) ===
IMG_DIR = '/content/img'
if os.path.exists(IMG_DIR_DRIVE) and not os.path.exists(IMG_DIR):
    print('Copying img/ to /content (one-time)...')
    shutil.copytree(IMG_DIR_DRIVE, IMG_DIR)
    print(f'Copied {len(os.listdir(IMG_DIR))} images.')
elif os.path.exists(IMG_DIR):
    print(f'Images already at /content/img ({len(os.listdir(IMG_DIR))} files).')
else:
    print('No img/ folder - multimodal booster will be skipped.')


## 3. Load test set

In [ ]:
def load_matching_exercises(path):
    exercises = []
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    for exam in data.get('exams', []):
        for ex in exam.get('exercises', []):
            if ex.get('type') != 'matching':
                continue
            exercises.append({
                'examId': exam.get('examId'),
                'level': exam.get('level'),
                'exerciseID': ex['exerciseID'],
                'instructions': ex.get('instructions', ''),
                'set1': ex['exercise']['set1'],
                'set2': ex['exercise']['set2'],
            })
    return exercises

def has_images(ex):
    return (any(o.get('image-path') for o in ex['set1'])
            or any(q.get('image-path') for q in ex['set2']))

exercises = load_matching_exercises(MATCHING_JSON)
img_exercise_ids = [ex['exerciseID'] for ex in exercises if has_images(ex)]
print(f'{len(exercises)} matching exercises | {len(img_exercise_ids)} have images')

## 4. Prompt + parsers

In [ ]:
import re
from collections import Counter

SYSTEM_PROMPT = (
    'Eres un experto en comprensión lectora del español, especializado en exámenes '
    'de proficiencia lingüística (Instituto Cervantes). Resuelves ejercicios de tipo '
    '"matching" leyendo cuidadosamente cada texto y eligiendo la mejor correspondencia. '
    'Prioriza coincidencias literales y específicas sobre paráfrasis vagas.'
)

def render_item(item):
    text = (item.get('text') or '').strip()
    img = item.get('image-path') or ''
    if img and text:
        return f"{text}\n[NOTA: este elemento incluye una imagen ({img}) que no puedes ver; usa el texto para decidir.]"
    if img and not text:
        return f"[Este elemento es solo una imagen ({img}) que no puedes ver. Elige la opción que parezca más probable según el contexto.]"
    return text

def build_user_message(ex):
    set1_ids = [str(item['optionId']) for item in ex['set1']]
    set2_ids = [str(item['optionId']) for item in ex['set2']]
    set1_block = '\n\n'.join(f"[{item['optionId']}]\n{render_item(item)}" for item in ex['set1'])
    set2_block = '\n'.join(f"[{item['optionId']}] {render_item(item)}" for item in ex['set2'])
    json_template = '{' + ', '.join(f'"{i}": "<letra>"' for i in set2_ids) + '}'
    return (
        f"INSTRUCCIONES DEL EJERCICIO:\n{ex['instructions']}\n\n"
        f"TEXTOS DEL SET 1 (identificadores válidos: {', '.join(set1_ids)}):\n{set1_block}\n\n"
        f"PREGUNTAS DEL SET 2 (identificadores: {', '.join(set2_ids)}):\n{set2_block}\n\n"
        f"TAREA: Para cada pregunta del Set 2, decide qué texto del Set 1 le corresponde mejor. "
        f"Razona brevemente y, al FINAL de tu respuesta, escribe SOLO una línea con un JSON "
        f"que asocie cada identificador del Set 2 con un identificador del Set 1.\n\n"
        f"REGLAS ESTRICTAS:\n"
        f"- Cada respuesta DEBE ser una de estas letras: {', '.join(set1_ids)}.\n"
        f"- Nunca uses '-', 'ninguno' ni vacíos. Si dudas, elige la opción más probable.\n\n"
        f"Formato exacto del JSON final (una sola línea, sin texto extra después):\n"
        f"{json_template}"
    )

JSON_OBJ_RE = re.compile(r'\{[^{}]*\}', re.DOTALL)

def parse_json_validated(text, set2_ids, valid_letters):
    for cand in reversed(JSON_OBJ_RE.findall(text)):
        try:
            obj = json.loads(cand)
        except Exception:
            continue
        norm = {str(k): str(v).strip().upper() for k, v in obj.items()}
        if not all(qid in norm for qid in set2_ids):
            continue
        out = {qid: (norm[qid] if norm[qid] in valid_letters else None) for qid in set2_ids}
        if any(v is not None for v in out.values()):
            return out
    return {qid: None for qid in set2_ids}

def fallback_lookup(text, qid, valid_letters):
    for m in re.finditer(rf'["\']?{re.escape(qid)}["\']?', text):
        for ch in text[m.end(): m.end() + 200]:
            if ch.upper() in valid_letters:
                return ch.upper()
    return None

def parse_response(text, set2_ids, set1_ids):
    valid = set(set1_ids)
    parsed = parse_json_validated(text, set2_ids, valid)
    final = {}
    for qid in set2_ids:
        choice = parsed.get(qid) or fallback_lookup(text, qid, valid) or set1_ids[0]
        final[qid] = choice
    return final

def detect_degenerate(predictions, min_questions=5):
    """An exercise is degenerate when ≥(N-1) of its N answers are the same letter."""
    bad = []
    for eid, p in predictions.items():
        if len(p) < min_questions:
            continue
        most_common, count = Counter(p.values()).most_common(1)[0]
        if count >= len(p) - 1:
            bad.append(eid)
    return bad

def save_predictions(predictions):
    with open(LOCAL_PRED, 'w', encoding='utf-8') as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)
    try:
        shutil.copy(LOCAL_PRED, DRIVE_PRED)
    except Exception as e:
        print('  (Drive backup failed, but local save OK:', e, ')')

print('Helpers ready.')

## 5. Load text-only LLM (Qwen2.5-7B)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb,
    device_map='auto', low_cpu_mem_usage=True,
)
model.eval()
print('Text model loaded:', MODEL_NAME)

## 6. Auto-detect degenerate exercises and queue for re-do

Identifies exercises where the parser fell back to the same letter (e.g., all `A` or all `E`). The 3 currently known are `A1_2015-11-21_E3`, `A2_2013-11-23_E1`, `A2_2014-11-01_E4` — the auto-detector should pick those up. Add IDs to `EXTRA_REDO` if you want to force-redo any other exercise.

In [ ]:
predictions = {}
if os.path.exists(LOCAL_PRED):
    with open(LOCAL_PRED, 'r', encoding='utf-8') as f:
        predictions = json.load(f)
print(f'Loaded {len(predictions)} existing predictions.')

EXTRA_REDO = set()  # add exerciseIDs to force a re-run

degenerate = set(detect_degenerate(predictions)) | EXTRA_REDO
print(f'\n{len(degenerate)} exercise(s) flagged for re-run:')
for eid in sorted(degenerate):
    print('  -', eid, '->', predictions.get(eid))

for eid in degenerate:
    predictions.pop(eid, None)
save_predictions(predictions)
print(f'\n{len(predictions)} predictions kept; {len(degenerate)} removed for re-run.')

## 7. Re-run missing exercises (text-only, N=3 self-consistency)

This includes the degenerate ones we just removed. Anything not in the predictions dict gets solved here.

In [ ]:
import time

N_SAMPLES = 3
MAX_NEW_TOKENS = 1500

@torch.inference_mode()
def generate(messages, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=0.4):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else 1.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = out[0, inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

def solve_text(ex, n_samples=N_SAMPLES):
    set1_ids = [str(o['optionId']) for o in ex['set1']]
    set2_ids = [str(q['optionId']) for q in ex['set2']]
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': build_user_message(ex)},
    ]
    if n_samples == 1:
        return parse_response(generate(messages, do_sample=False), set2_ids, set1_ids)
    preds = []
    for i in range(n_samples):
        raw = generate(messages, do_sample=(i > 0), temperature=0.4)
        preds.append(parse_response(raw, set2_ids, set1_ids))
    final = {}
    for qid in set2_ids:
        votes = Counter(p[qid] for p in preds)
        final[qid] = votes.most_common(1)[0][0]
    return final

remaining = [ex for ex in exercises if ex['exerciseID'] not in predictions]
print(f'{len(remaining)} exercise(s) to solve | N_SAMPLES = {N_SAMPLES}')

t0 = time.time()
for i, ex in enumerate(remaining, 1):
    pred = solve_text(ex, N_SAMPLES)
    predictions[ex['exerciseID']] = pred
    save_predictions(predictions)
    elapsed = time.time() - t0
    avg = elapsed / i
    eta = avg * (len(remaining) - i)
    print(f"[{i}/{len(remaining)}] {ex['exerciseID']} ({ex['level']}) | "
          f'avg {avg:.1f}s/ex | ETA {eta/60:.1f} min')

print(f'\nText pass done. {len(predictions)} total predictions.')

## 8. Free text model and load Qwen2.5-VL-7B (multimodal)

In [ ]:
import gc
try:
    del model, tokenizer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print('GPU mem free:', torch.cuda.mem_get_info()[0] / 1e9, 'GB')

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

VL_MODEL_NAME = 'Qwen/Qwen2.5-VL-7B-Instruct'
vl_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VL_MODEL_NAME, quantization_config=bnb,
    device_map='auto', low_cpu_mem_usage=True,
)
vl_model.eval()
vl_processor = AutoProcessor.from_pretrained(VL_MODEL_NAME)
print('VL model loaded:', VL_MODEL_NAME)

## 9. Surgical multimodal pass — ONLY exercises where SET1 has images

v2 lesson: blanket VL overwrites hurt. v3 only touches exercises where the *content side* (SET1) is pictures and text-only is literally guessing.
Uses a fresh `vl_done_v3.json` marker so prior runs don't interfere.


In [ ]:
import gc
from PIL import Image

def img_path_local(rel):
    """Convert dataset image-path (e.g. 'img/foo.jpg') to a real local file path."""
    if not rel:
        return None
    base = os.path.basename(rel)
    p = os.path.join(IMG_DIR, base)
    if os.path.exists(p):
        return p
    p2 = os.path.join(TEST_SET_DIR, rel)
    return p2 if os.path.exists(p2) else None

def has_set1_images(ex):
    """True only when the option/content side has at least one real image."""
    return any(img_path_local(o.get('image-path')) for o in ex['set1'])

MAX_SIDE = 512  # bigger than v2 (384) — SET1 images ARE the content; we need to read them

def load_capped_image(path):
    img = Image.open(path).convert('RGB')
    w, h = img.size
    scale = MAX_SIDE / max(w, h)
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return img

def build_vl_messages(ex):
    set1_ids = [str(item['optionId']) for item in ex['set1']]
    set2_ids = [str(item['optionId']) for item in ex['set2']]
    json_template = '{' + ', '.join(f'"{i}": "<letra>"' for i in set2_ids) + '}'
    content = [{'type': 'text', 'text': f"INSTRUCCIONES: {ex['instructions']}\n\nSET 1:"}]
    for item in ex['set1']:
        content.append({'type': 'text', 'text': f"\n[{item['optionId']}]"})
        ip = img_path_local(item.get('image-path'))
        if ip: content.append({'type': 'image', 'image': load_capped_image(ip)})
        if item.get('text'): content.append({'type': 'text', 'text': item['text'].strip()})
    content.append({'type': 'text', 'text': '\n\nSET 2:'})
    for item in ex['set2']:
        content.append({'type': 'text', 'text': f"\n[{item['optionId']}]"})
        ip = img_path_local(item.get('image-path'))
        if ip: content.append({'type': 'image', 'image': load_capped_image(ip)})
        if item.get('text'): content.append({'type': 'text', 'text': item['text'].strip()})
    content.append({'type': 'text', 'text':
        f"\n\nTAREA: para cada pregunta del Set 2 elige la opción del Set 1 que mejor le corresponda. "
        f"Cada respuesta debe ser una de: {', '.join(set1_ids)}. "
        f"Nunca uses '-' o vacíos. Razona brevemente y al final escribe SOLO el JSON: {json_template}"
    })
    return [{'role': 'user', 'content': content}]

@torch.inference_mode()
def vl_generate(messages, max_new_tokens=600, do_sample=False, temperature=0.4):
    text = vl_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = vl_processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors='pt',
    ).to(vl_model.device)
    out = vl_model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=do_sample, temperature=temperature if do_sample else 1.0,
    )
    new_tokens = out[0, inputs.input_ids.shape[-1]:]
    return vl_processor.decode(new_tokens, skip_special_tokens=True)

def solve_vl(ex, n_samples=1):
    set1_ids = [str(o['optionId']) for o in ex['set1']]
    set2_ids = [str(q['optionId']) for q in ex['set2']]
    messages = build_vl_messages(ex)
    if n_samples == 1:
        return parse_response(vl_generate(messages, do_sample=False), set2_ids, set1_ids)
    preds = []
    for i in range(n_samples):
        raw = vl_generate(messages, do_sample=(i > 0), temperature=0.4)
        preds.append(parse_response(raw, set2_ids, set1_ids))
    final = {}
    for qid in set2_ids:
        votes = Counter(p[qid] for p in preds)
        final[qid] = votes.most_common(1)[0][0]
    return final

# ── v3 marker (fresh — independent of v2's vl_done.json)
VL_DONE_PATH = os.path.join(os.path.dirname(LOCAL_PRED), 'vl_done_v3.json')
vl_done = set()
if os.path.exists(VL_DONE_PATH):
    with open(VL_DONE_PATH) as f:
        vl_done = set(json.load(f))
print(f'Already VL-processed in v3: {len(vl_done)}')

VL_N_SAMPLES = 1
set1_img_ex = [ex for ex in exercises if has_set1_images(ex)]
todo = [ex for ex in set1_img_ex if ex['exerciseID'] not in vl_done]
print(f'Surgical VL pass: {len(set1_img_ex)} exercises have SET1 images; {len(todo)} to do (N={VL_N_SAMPLES})')
print('Targets:')
for ex in set1_img_ex:
    print(f"  - {ex['exerciseID']} ({ex['level']}) | set1 size={len(ex['set1'])} | set2 size={len(ex['set2'])}")

t0 = time.time()
for i, ex in enumerate(todo, 1):
    try:
        pred = solve_vl(ex, n_samples=VL_N_SAMPLES)
        predictions[ex['exerciseID']] = pred  # overwrite text-only prediction
        save_predictions(predictions)
        vl_done.add(ex['exerciseID'])
        with open(VL_DONE_PATH, 'w') as f:
            json.dump(sorted(vl_done), f)
        status = 'ok'
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        status = 'OOM-skipped (kept text-only)'
    except Exception as e:
        gc.collect(); torch.cuda.empty_cache()
        status = f'err {type(e).__name__}-skipped'

    elapsed = time.time() - t0
    avg = elapsed / i
    eta = avg * (len(todo) - i)
    print(f"[{i}/{len(todo)}] {ex['exerciseID']} ({ex['level']}) | {status} | "
          f'avg {avg:.1f}s | ETA {eta/60:.1f} min')

print('\nSurgical multimodal pass done.')


## 10. Final sanity check

In [ ]:
issues = []
for ex in exercises:
    eid = ex['exerciseID']
    set1_ids = {str(o['optionId']) for o in ex['set1']}
    set2_ids = [str(q['optionId']) for q in ex['set2']]
    pred = predictions.get(eid)
    if pred is None:
        issues.append(f'MISSING {eid}'); continue
    for qid in set2_ids:
        if qid not in pred:
            issues.append(f'{eid}: missing q{qid}')
        elif pred[qid] not in set1_ids:
            issues.append(f'{eid}: invalid value {pred[qid]} for q{qid}')

still_degenerate = detect_degenerate(predictions)
if issues:
    print(f'{len(issues)} format issue(s):')
    for i in issues[:20]: print(' ', i)
else:
    n_q = sum(len(v) for v in predictions.values())
    print(f'OK · {len(predictions)} exercises · {n_q} answers · all valid letters.')
if still_degenerate:
    print(f'\nStill-degenerate exercises ({len(still_degenerate)}):')
    for d in still_degenerate: print(' ', d, '->', predictions[d])

## 11. Build the submission zip

In [ ]:
shutil.copy(LOCAL_PRED, '/content/matching_preds.json')
import zipfile
with zipfile.ZipFile('/content/profe2026_v3.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('/content/matching_preds.json', 'matching_preds.json')
print('Wrote /content/profe2026_v3.zip')
with zipfile.ZipFile('/content/profe2026_v3.zip') as z:
    print('Zip contents:', z.namelist())

from google.colab import files as _files
_files.download('/content/profe2026_v3.zip')
